In [3]:
import pandas as pd
import numpy as np
import gc

In [4]:
cal = pd.read_csv('calendar.csv')
val = pd.read_csv('sales_train_validation.csv')
eva = pd.read_csv('sales_train_evaluation.csv')
pr = pd.read_csv('sell_prices.csv')

due to constant memory crashes, two solutions were implemented \
i. downcasting dtypes\
ii. using garbage collection (del and gc.collect())

In [5]:
def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    for i, t in enumerate(types):
        if 'int' in str(t):
            if df[cols[i]].min() > np.iinfo(np.int8).min and df[cols[i]].max() < np.iinfo(np.int8).max:
                df[cols[i]] = df[cols[i]].astype(np.int8)
            elif df[cols[i]].min() > np.iinfo(np.int16).min and df[cols[i]].max() < np.iinfo(np.int16).max:
                df[cols[i]] = df[cols[i]].astype(np.int16)
            elif df[cols[i]].min() > np.iinfo(np.int32).min and df[cols[i]].max() < np.iinfo(np.int32).max:
                df[cols[i]] = df[cols[i]].astype(np.int32)
        elif 'float' in str(t):
            if df[cols[i]].min() > np.finfo(np.float16).min and df[cols[i]].max() < np.finfo(np.float16).max:
                df[cols[i]] = df[cols[i]].astype(np.float16)
            elif df[cols[i]].min() > np.finfo(np.float32).min and df[cols[i]].max() < np.finfo(np.float32).max:
                df[cols[i]] = df[cols[i]].astype(np.float32)
        elif t == object:
            df[cols[i]] = df[cols[i]].astype('category')
    return df

In [6]:
val = downcast(val)
cal = downcast(cal)
pr = downcast(pr)

In [7]:
print(val.columns.tolist())

['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14', 'd_15', 'd_16', 'd_17', 'd_18', 'd_19', 'd_20', 'd_21', 'd_22', 'd_23', 'd_24', 'd_25', 'd_26', 'd_27', 'd_28', 'd_29', 'd_30', 'd_31', 'd_32', 'd_33', 'd_34', 'd_35', 'd_36', 'd_37', 'd_38', 'd_39', 'd_40', 'd_41', 'd_42', 'd_43', 'd_44', 'd_45', 'd_46', 'd_47', 'd_48', 'd_49', 'd_50', 'd_51', 'd_52', 'd_53', 'd_54', 'd_55', 'd_56', 'd_57', 'd_58', 'd_59', 'd_60', 'd_61', 'd_62', 'd_63', 'd_64', 'd_65', 'd_66', 'd_67', 'd_68', 'd_69', 'd_70', 'd_71', 'd_72', 'd_73', 'd_74', 'd_75', 'd_76', 'd_77', 'd_78', 'd_79', 'd_80', 'd_81', 'd_82', 'd_83', 'd_84', 'd_85', 'd_86', 'd_87', 'd_88', 'd_89', 'd_90', 'd_91', 'd_92', 'd_93', 'd_94', 'd_95', 'd_96', 'd_97', 'd_98', 'd_99', 'd_100', 'd_101', 'd_102', 'd_103', 'd_104', 'd_105', 'd_106', 'd_107', 'd_108', 'd_109', 'd_110', 'd_111', 'd_112', 'd_113', 'd_114', 'd_115', 'd_116', '

In [8]:
print(cal.columns.tolist())

['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


In [9]:
print(pr.columns.tolist())

['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


In [10]:
melted_val = pd.melt(val,id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], var_name='d', value_name='sales')
del val
gc.collect()

0

In [11]:
melted_val.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [12]:
cols = ['d', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_type_1', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']  # wday in cal 1-7 => sat to fri
train_df = melted_val.merge(cal[cols], on="d", how="left")
del melted_val, cal
gc.collect()

0

In [13]:
train_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,wday,month,year,event_type_1,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0


In [14]:
train_df = train_df.merge(pr, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del pr
gc.collect()

0

In [15]:
train_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,wday,month,year,event_type_1,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,0,0,0,NaN


In [16]:
train_df.dtypes

,0
id,category
item_id,category
dept_id,category
cat_id,category
store_id,category
state_id,category
d,object
sales,int16
date,category
wm_yr_wk,int16


In [17]:
train_df = train_df.drop(columns=['id'])

In [18]:
train_df['d'] = train_df['d'].str[2:].astype(int)

In [19]:
train_df['wm_yr_wk'] = train_df['wm_yr_wk']%100

train_df['date'] = pd.to_datetime(train_df['date'])
train_df['date'] = train_df['date'].dt.day

train_df = train_df.rename(columns = {'wm_yr_wk': 'wno', 'date': 'day'})

In [20]:
print(train_df['year'].unique())

[2011 2012 2013 2014 2015 2016]


In [21]:
train_df['year'] = train_df['year'] - train_df['year'].min()  # 0 is 2011, 1 is 2012 and so on

In [22]:
train_df['has_event'] = train_df['event_type_1'].notna().astype(int)

# only if type_1 is there then type_2 exists verified with cal[(cal['event_name_1'].isna()) & (cal['event_name_2'].notna())].shape which gives (0,14)

In [23]:
print(train_df[(train_df['snap_CA']==1) & (train_df['snap_WI']==1) & (train_df['snap_TX']==1)])

                item_id    dept_id   cat_id store_id state_id     d  sales  \
152450    HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1       CA     6      0   
152451    HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1       CA     6      0   
152452    HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1       CA     6      0   
152453    HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1       CA     6      0   
152454    HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1       CA     6      0   
...                 ...        ...      ...      ...      ...   ...    ...   
57870015    FOODS_3_823    FOODS_3    FOODS     WI_3       WI  1898      0   
57870016    FOODS_3_824    FOODS_3    FOODS     WI_3       WI  1898      1   
57870017    FOODS_3_825    FOODS_3    FOODS     WI_3       WI  1898      2   
57870018    FOODS_3_826    FOODS_3    FOODS     WI_3       WI  1898      1   
57870019    FOODS_3_827    FOODS_3    FOODS     WI_3       WI  1898      1   

          day  wno  wday  month  year event_type_1 event_type_2

In [24]:
conditions = [(train_df['state_id'] == 'CA'), (train_df['state_id'] == 'TX'), (train_df['state_id'] == 'WI')]
choices = [train_df['snap_CA'], train_df['snap_TX'], train_df['snap_WI']]
train_df['snap']=np.select(conditions, choices, default=0)
train_df = train_df.drop(columns=['snap_CA', 'snap_TX', 'snap_WI'])

In [25]:
print(train_df[(train_df['sell_price'].isna()) & (train_df['sales']!=0)])

Empty DataFrame
Columns: [item_id, dept_id, cat_id, store_id, state_id, d, sales, day, wno, wday, month, year, event_type_1, event_type_2, sell_price, has_event, snap]
Index: []


In [26]:
train_df['sell_price'] = train_df['sell_price'].fillna(0)

In [27]:
train_df = downcast(train_df)

In [28]:
cols=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'day', 'wday', 'wno', 'month', 'year', 'has_event', 'event_type_1', 'event_type_2', 'snap', 'sell_price', 'sales']
train_df = train_df[cols]

In [29]:
train_df.head()

,item_id,dept_id,cat_id,store_id,state_id,d,day,wday,wno,month,year,has_event,event_type_1,event_type_2,snap,sell_price,sales
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,0.0,0
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,0.0,0
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,0.0,0
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,0.0,0
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,0.0,0


In [30]:
train_df.dtypes

,0
item_id,category
dept_id,category
cat_id,category
store_id,category
state_id,category
d,int16
day,int8
wday,int8
wno,int8
month,int8


In [31]:
train_df = train_df.sort_values(by=['item_id', 'store_id', 'd'])

In [32]:
train_df.head()

,item_id,dept_id,cat_id,store_id,state_id,d,day,wday,wno,month,year,has_event,event_type_1,event_type_2,snap,sell_price,sales
1612,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,29,1,1,1,0,0,NaN,NaN,0,2.0,3
32102,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,2,30,2,1,1,0,0,NaN,NaN,0,2.0,0
62592,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,31,3,1,1,0,0,NaN,NaN,0,2.0,0
93082,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,4,1,4,1,2,0,0,NaN,NaN,1,2.0,1
123572,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,5,2,5,1,2,0,0,NaN,NaN,1,2.0,4


In [33]:
train_df.to_parquet('train_df_preprocessed.parquet', index=False)